# Ghost following agent

iMD agent that tries to move the molecule to match a ghost molecule placed with controllers.

<video controls src="./assets/ghost_follower.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation
from nanover.trajectory import FrameData
from nanover.mdanalysis import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("trypsin_benzamidine.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: ghost follower")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_interaction_modes()
utilities.use_transform_handles()

In [3]:
structure_atoms = universe.select_atoms("not resname BEN")
molecule_atoms = universe.select_atoms("resname BEN and not name H*")

utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=universe.select_atoms("resname BEN").atoms.indices)

Add objects to scene:

In [4]:
ghost = utilities.make_ghost_from_mdanalysis("molecule", atoms=molecule_atoms)

## Ghost follower

In [5]:
import numpy as np
from nanover.jupyter import ImdAgent
from nanover.imd import ParticleInteraction

class GhostFollowerAgent(ImdAgent):
    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        # target positions are original ghost positions transformed by ghost transform
        target_positions = utilities.transforms.fetch_transform(ghost.key).points_local_to_parent(ghost.positions)
        real_positions = full_frame.particle_positions[molecule_atoms.indices]

        target_centroid = target_positions.mean(axis=0)
        real_centroid = real_positions.mean(axis=0)

        utilities.objects.update_line(f"follow.centroid", positions=[real_centroid, target_centroid], size=0.01, color=[1.0, 0, 0, 1.0])
        self.interactions.update_interaction("follow.centroid", ParticleInteraction(
            position=target_centroid,
            particles=[int(x) for x in molecule_atoms.indices],
            type="spring",
            scale=500,
            max_force=100,
        ))

        # rotational following if centroid is close enough
        close = np.linalg.norm(real_centroid - target_centroid, axis=0) < 1

        # find target positions ignoring centroid differences
        rotational_target = target_positions - target_centroid
        rotational_real = real_positions - real_centroid
        rotational = real_positions + (rotational_target - rotational_real)

        for i, index in enumerate(molecule_atoms.indices):
            if close:
                utilities.objects.update_line(f"follow.{i}", positions=[rotational[i], real_positions[i]], size=0.01, color=[1.0, 0, 0, 1.0])
                self.interactions.update_interaction(f"follow.{i}", ParticleInteraction(
                    position=rotational[i],
                    particles=[int(index)],
                    type="spring",
                    scale=100,
                    max_force=50,
                ))
            else:
                utilities.objects.remove_line(f"follow.{i}")
                self.interactions.remove_interaction(f"follow.{i}")

follower = GhostFollowerAgent.from_runner(imd_runner)
follower.start()